In [1]:
import sympy as sp

# ============================================================
# 1. Symbols & Time-Dependent Functions
# ============================================================
x, z, t = sp.symbols('x z t')
pi = sp.pi

nu, kappa, g, alpha, gamma = sp.symbols('nu kappa g alpha gamma', positive=True)
sigma, r, b = sp.symbols('sigma r b', positive=True)

# Modal amplitudes as functions of time
X = sp.Function('X')(t)
Y = sp.Function('Y')(t)
Z = sp.Function('Z')(t)

# ============================================================
# 2. Galerkin basis (Lorenz 1963 truncation)
# ============================================================
# Aspect ratio 'a' is assumed 1 for simplicity (square cells)
psi = X * sp.sin(pi*x) * sp.sin(pi*z)
theta = Y * sp.cos(pi*x) * sp.sin(pi*z) + Z * sp.sin(2*pi*z)

u_x = sp.diff(psi, z)
u_z = -sp.diff(psi, x)

# ============================================================
# 3. Boussinesq Equations
# ============================================================
# Temperature Eq: dT/dt + u.grad(T) = kappa * lap(T)
# T = T_background + theta, where T_background = T0 - gamma*z
theta_t = sp.diff(theta, t)
adv_theta = u_x * sp.diff(theta, x) + u_z * sp.diff(theta, z)
# Linear background gradient advection: u_z * d(T_bg)/dz = u_z * (-gamma)
bg_advection = -u_z * gamma 
lap_theta = sp.diff(theta, x, 2) + sp.diff(theta, z, 2)

temp_eq = theta_t + adv_theta + bg_advection - kappa * lap_theta

# Vorticity Eq: d/dt(lap(psi)) + J(psi, lap(psi)) = nu*lap4(psi) + g*alpha*d(theta)/dx
lap_psi = sp.diff(psi, x, 2) + sp.diff(psi, z, 2)
lap_psi_t = sp.diff(lap_psi, t)
jacobian = sp.diff(psi, x) * sp.diff(lap_psi, z) - sp.diff(psi, z) * sp.diff(lap_psi, x)
lap4_psi = sp.diff(lap_psi, x, 2) + sp.diff(lap_psi, z, 2)

vorticity_eq = lap_psi_t + jacobian - nu * lap4_psi - g * alpha * sp.diff(theta, x)

# ============================================================
# 4. Projection
# ============================================================
basis_X = sp.sin(pi*x)*sp.sin(pi*z)
basis_Y = sp.cos(pi*x)*sp.sin(pi*z)
basis_Z = sp.sin(2*pi*z)

def project(expr, basis):
    num = sp.integrate(sp.integrate(expr*basis, (x, 0, 1)), (z, 0, 1))
    den = sp.integrate(sp.integrate(basis*basis, (x, 0, 1)), (z, 0, 1))
    return sp.simplify(num/den)

# Solve for time derivatives
dXdt_raw = sp.solve(project(vorticity_eq, basis_X), sp.diff(X, t))[0]
dYdt_raw = sp.solve(project(temp_eq, basis_Y), sp.diff(Y, t))[0]
dZdt_raw = sp.solve(project(temp_eq, basis_Z), sp.diff(Z, t))[0]

# ============================================================
# 5. Dimensionless Scaling (Lorenz 1963)
# ============================================================
# We scale time by the thermal relaxation time of the mode
tau_scale = 2 * pi**2 * kappa 

# Scaling factors to transform (X, Y, Z) -> (x_var, y_var, z_var)
# These specific constants eliminate the messy pi and g*alpha terms
Cx = (sp.sqrt(2) * 2 * pi**2 * kappa) / pi 
Cy = (2 * pi**3 * kappa * nu) / (g * alpha * sp.sqrt(2))
Cz = (2 * pi**3 * kappa * nu) / (g * alpha)

x_v, y_v, z_v = sp.symbols('x y z')
dxdtau, dydtau, dzdtau = sp.symbols('dxdtau dydtau dzdtau')

# Chain Rule: dX/dt = (dX/dtau) * (dtau/dt) = Cx * dxdtau * tau_scale
eq1 = sp.Eq(dxdtau * Cx * tau_scale, dXdt_raw.subs({X: Cx*x_v, Y: Cy*y_v, Z: Cz*z_v}))
eq2 = sp.Eq(dydtau * Cy * tau_scale, dYdt_raw.subs({X: Cx*x_v, Y: Cy*y_v, Z: Cz*z_v}))
eq3 = sp.Eq(dzdtau * Cz * tau_scale, dZdt_raw.subs({X: Cx*x_v, Y: Cy*y_v, Z: Cz*z_v}))

# Solve for the dimensionless derivatives
res_dx = sp.solve(eq1, dxdtau)[0]
res_dy = sp.solve(eq2, dydtau)[0]
res_dz = sp.solve(eq3, dzdtau)[0]

# ============================================================
# 6. Final Substitution of Parameters
# ============================================================
# Prandtl number: sigma = nu / kappa
# Rayleigh number: r = (g * alpha * gamma) / (nu * kappa * k^3) -> Simplified for our mode
r_val = (g * alpha * gamma) / (nu * kappa * 2 * pi**4) 

final_dx = sp.simplify(res_dx.subs(nu, sigma * kappa))
final_dy = sp.simplify(res_dy.subs(gamma, r_val * (nu * kappa * 2 * pi**4) / (g * alpha)).subs(nu, sigma * kappa))
final_dz = sp.simplify(res_dz.subs(4, b)) # 'b' relates to the geometry (8/3 for square)

print("--- Final Dimensionless Lorenz System ---")
print(f"dx/dtau = {final_dx}")
print(f"dy/dtau = {final_dy}")
print(f"dz/dtau = {final_dz}")

--- Final Dimensionless Lorenz System ---
dx/dtau = -sigma*x + sigma*y/(8*pi)
dy/dtau = -alpha*g*gamma*x/(pi**3*kappa**2*sigma) - 2*pi*x*z - y
dz/dtau = pi*x*y/2 - 2*z
